# Experimento de 140 combinaciones, parte 1: diseño

Este cuaderno describe el diseño del experimento que pide la guía: 7 modelos de clasificación × 4 balanceos × 4 optimizadores,
más 7 modelos de regresión × 4 optimizadores. Aquí no se hace ningún entrenamiento pesado. Se explican las decisiones de diseño:
la validación espacial, los espacios de búsqueda y el presupuesto de cómputo de cada modelo.

## Flujo del experimento

![Flujo del experimento](fig_pipeline_experimento.png)

## 1. Qué se compara

| Tarea | Modelos | Balanceo | Optimizadores | Combinaciones |
|---|---|---|---|---|
| Clasificación de la clase de aptitud (Baja, Media, Alta) | KNN, Naive Bayes, regresión logística L1/L2, árbol de decisión, Random Forest, XGBoost, SVM RBF | ninguno, SMOTE, ADASYN, `class_weight` | grid search, random search, optimización bayesiana, algoritmo genético | 112 |
| Regresión del índice de aptitud | KNN, Ridge, Lasso, árbol de decisión, Random Forest, XGBoost, SVR RBF | no aplica | los mismos 4 | 28 |

De las 112 de clasificación, 4 no aplican: KNN con `class_weight`. KNN no minimiza una función de pérdida que se pueda ponderar por clase,
así que se registra como no aplicable en vez de inventar un equivalente. Quedan 108 + 28 = 136 combinaciones que se corren.

Aparte, y sin contar en las 140, se corre un método de multifidelidad, el successive halving, que la guía recomienda.

En las tablas de los cuadernos siguientes los optimizadores aparecen con estos nombres: `grilla` (grid search), `aleatoria` (random search), `bayesiana` (optimización bayesiana con Optuna) y `genetica` (algoritmo genético con DEAP).

## 2. Validación anidada por bloques espaciales

- Bucle externo, 5 pliegues. Solo mide. Cada pliegue deja fuera bloques enteros de 5° × 5°, con `StratifiedGroupKFold`.
  Con partición aleatoria, plantas vecinas quedan a los dos lados y el error sale optimista (Roberts et al., 2017).
  Se estratifica por clase para que la clase Baja (3 %) aparezca en cada prueba.
- Bucle interno, 3 pliegues. Elige hiperparámetros dentro del entrenamiento externo, también por bloques. Se calculan
  una vez por pliegue externo y los usan los cuatro optimizadores, así cada candidato se evalúa con las mismas particiones.
- Si se eligen hiperparámetros y se mide con los mismos datos, el error sale optimista. Esto ya está documentado
  (Varma y Simon, 2006; Cawley y Talbot, 2010) y por eso las métricas que se reportan salen solo del bucle externo.
- 5 × 3 es un compromiso de costo. Con 30 evaluaciones por búsqueda, cada combinación ajusta 5 × (3 × 30 + 1) = 455 modelos.

Todo lo que aprende de los datos (la mediana que imputa, la media y desviación que escalan, los puntos sintéticos de SMOTE y ADASYN)
va dentro del pipeline y se ajusta solo con el entrenamiento de cada pliegue.

In [1]:
import numpy as np
import pandas as pd
from src import balancing, config, cv, data, models, runner
from src.search.espacio import grilla, niveles_grilla
from src.search.halving import plan_rondas

pd.set_option("display.max_colwidth", 120)
b = data.cargar()
print(f"Filas: {len(b.y_clf):,} | predictoras: {b.X.shape[1]} | bloques: {len(set(b.grupos))}")

filas = []
for k, (tr, te) in enumerate(cv.externas(b.y_clf, b.grupos)):
    cuenta = np.bincount(b.y_clf[te], minlength=3)
    filas.append({"pliegue": k, "filas de prueba": len(te), "bloques de prueba": len(set(b.grupos[te])),
                  **{f"{c} en prueba": n for c, n in zip(config.CLASES, cuenta)},
                  "bloques compartidos con entrenamiento": len(set(b.grupos[tr]) & set(b.grupos[te]))})
pd.DataFrame(filas).set_index("pliegue")

C:\Users\Enma\miniconda3\envs\solarpv-eda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Filas: 57,976 | predictoras: 15 | bloques: 496


,filas de prueba,bloques de prueba,Baja en prueba,Media en prueba,Alta en prueba,bloques compartidos con entrenamiento
pliegue,,,,,,
0,11571,93,339,2552,8680,0
1,11573,89,344,2551,8678,0
2,11673,106,445,2550,8678,0
3,11586,107,358,2550,8678,0
4,11573,101,340,2553,8680,0


La última columna en cero es la garantía de que ningún bloque queda partido. La clase Baja tiene casos en los 5 pliegues de prueba.

## 3. Modelos y espacios de búsqueda

Los rangos salen de Probst, Boulesteix y Bischl (2019), que midieron en 38 conjuntos de datos qué hiperparámetros vale la pena ajustar
y en qué rangos, para árboles, bosques, XGBoost, SVM, kNN y modelos lineales regularizados. Para el SVM se sigue además la guía de
Hsu, Chang y Lin (2003): C y gamma en escala logarítmica. C, alpha y la tasa de aprendizaje van en escala logarítmica en todos los casos.

Tres recortes por costo, medidos en un piloto (sección 6):

- El número de árboles de Random Forest se acota a 200 y el de XGBoost a 300. Probst et al. muestran que el número de árboles casi
  no gana con el ajuste: más árboles casi nunca empeoran, solo cuestan más.
- KNN usa distancia euclidiana. Manhattan dio el mismo F1 en el piloto y es 16 veces más lenta.
- El SVM con kernel RBF se entrena en una submuestra estratificada de 5,000 filas. Su costo crece entre n² y n³.

In [2]:
tipo = {"float": "real", "int": "entero", "cat": "categórico"}
def describe(spec):
    if spec[0] == "cat":
        return f"{{{', '.join(map(str, spec[1]))}}}"
    return f"[{spec[1]:g}, {spec[2]:g}]" + (", log" if spec[3] else "")

filas = []
for tarea in ["clf", "reg"]:
    for m in models.MODELOS[tarea]:
        esp = models.ESPACIOS[tarea][m]
        niv = niveles_grilla(esp, config.PRESUPUESTO)
        for p, spec in esp.items():
            filas.append({"tarea": tarea, "modelo": m, "hiperparámetro": p, "tipo": tipo[spec[0]], "rango": describe(spec),
                          "niveles en la grilla": niv[p]})
espacios = pd.DataFrame(filas)
espacios.set_index(["tarea", "modelo", "hiperparámetro"])

tipo  \
tarea modelo              hiperparámetro                  
clf   KNN                 n_neighbors            entero   
                          weights            categórico   
      Naive Bayes         var_smoothing            real   
      Regresión logística C                        real   
                          l1_ratio           categórico   
      Árbol de decisión   max_depth              entero   
                          min_samples_leaf       entero   
                          min_samples_split      entero   
                          max_features             real   
      Random Forest       n_estimators           entero   
                          max_features             real   
                          min_samples_leaf       entero   
                          max_samples              real   
      XGBoost             n_estimators           entero   
                          learning_rate            real   
                          max_depth              entero   
                          min_child_weight         real   
                          subsample                real   
                          colsample_bytree         real   
      SVM RBF             C                        real   
                          gamma                    real   
reg   KNN                 n_neighbors            entero   
                          weights            categórico   
      Ridge               alpha                    real   
      Lasso               alpha                    real   
      Árbol de decisión   max_depth              entero   
                          min_samples_leaf       entero   
                          min_samples_split      entero   
                          max_features             real   
      Random Forest       n_estimators           entero   
                          max_features             real   
                          min_samples_leaf       entero   
                          max_samples              real   
      XGBoost             n_estimators           entero   
                          learning_rate            real   
                          max_depth              entero   
                          min_child_weight         real   
                          subsample                real   
                          colsample_bytree         real   
      SVR RBF             C                        real   
                          gamma                    real   
                          epsilon                  real   

                                                                rango  \
tarea modelo              hiperparámetro                                
clf   KNN                 n_neighbors                    [1, 60], log   
                          weights                 {uniform, distance}   
      Naive Bayes         var_smoothing               [1e-12, 1], log   
      Regresión logística C                  [0.000976562, 1024], log   
                          l1_ratio                         {0.0, 1.0}   
      Árbol de decisión   max_depth                           [1, 30]   
                          min_samples_leaf               [1, 60], log   
                          min_samples_split              [2, 60], log   
                          max_features                       [0.1, 1]   
      Random Forest       n_estimators                      [50, 200]   
                          max_features                       [0.1, 1]   
                          min_samples_leaf               [1, 50], log   
                          max_samples                        [0.1, 1]   
      XGBoost             n_estimators                      [50, 300]   
                          learning_rate         [0.000976562, 1], log   
                          max_depth                           [1, 15]   
                          min_child_weight              [1, 128], log   
                          subsample                          [0.1, 

In [3]:
pd.DataFrame([{"tarea": t, "modelo": m, "puntos de la grilla": len(grilla(models.ESPACIOS[t][m], config.PRESUPUESTO)),
               "hiperparámetros": len(models.ESPACIOS[t][m])}
              for t in ["clf", "reg"] for m in models.MODELOS[t]]).set_index(["tarea", "modelo"]).T

tarea               clf                                                    \
modelo              KNN Naive Bayes Regresión logística Árbol de decisión   
puntos de la grilla  26          30                  30                24   
hiperparámetros       2           1                   2                 4   

tarea                                             reg              \
modelo              Random Forest XGBoost SVM RBF KNN Ridge Lasso   
puntos de la grilla            24      24      30  26    30    30   
hiperparámetros                 4       6       2   2     1     1   

tarea                                                                
modelo              Árbol de decisión Random Forest XGBoost SVR RBF  
puntos de la grilla                24            24      24      27  
hiperparámetros                     4             4       6       3

El grid search no siempre llena las 30 evaluaciones. Con 4 hiperparámetros, el producto de niveles que cabe es 24 (3 × 2 × 2 × 2); con 6, varios
quedan en un solo valor, el centro del rango. Es la debilidad conocida del grid search: gasta evaluaciones repitiendo valores de los
hiperparámetros que importan poco (Bergstra y Bengio, 2012). Se deja así y se reporta el número real de evaluaciones de cada búsqueda.

## 4. Balanceo

| Estrategia | Qué hace | Cómo se aplica |
|---|---|---|
| Ninguno | Nada | Los datos tal cual |
| SMOTE | Crea puntos sintéticos de la clase minoritaria interpolando entre vecinos (Chawla et al., 2002) | Paso del `imblearn.pipeline.Pipeline`, solo en `fit` |
| ADASYN | Como SMOTE, pero crea más puntos donde la minoritaria es más difícil de aprender (He et al., 2008) | Igual |
| `class_weight` | No crea datos: pesa cada clase por el inverso de su frecuencia en la pérdida | Nativo en logística, árbol, bosque y SVM; `sample_weight` equivalente en Naive Bayes y XGBoost; no aplica en KNN |

`imblearn` garantiza que el remuestreo ocurre solo al entrenar (Lemaître et al., 2017): validación y prueba nunca ven puntos sintéticos.
Se escala antes de remuestrear porque SMOTE y ADASYN buscan vecinos por distancia euclidiana. Con latitud y longitud entre las variables, SMOTE puede crear puntos sintéticos entre dos regiones distintas.

## 5. Optimizadores

Los cuatro buscan en el mismo espacio. Todo pasa por un cubo unitario [0, 1]^d que se decodifica a valores reales: el grid search toma los
centros de celda, el random search muestrea el cubo uniforme, el algoritmo genético evoluciona puntos del cubo y Optuna usa las mismas
escalas con sus `suggest_*`. Todos llaman al mismo evaluador, que ajusta el pipeline completo en cada pliegue interno y cuenta el
presupuesto en evaluaciones distintas: si un optimizador propone un candidato repetido, sale del caché y no gasta presupuesto.

| Optimizador | Componentes declarados |
|---|---|
| Grid search (`grilla`) | Centros de celda; los niveles se reparten para que el producto no pase de 30 |
| Random search (`aleatoria`) | 30 puntos uniformes en el cubo; con el mismo presupuesto prueba más valores distintos de cada hiperparámetro que el grid search (Bergstra y Bengio, 2012) |
| Optimización bayesiana (Optuna, `bayesiana`) | Sustituto: TPE, *Tree-structured Parzen Estimator* (Bergstra et al., 2011). Adquisición: mejora esperada, que con TPE equivale a maximizar l(x)/g(x); se comparan 24 candidatos por paso. Las primeras 10 evaluaciones son aleatorias (Akiba et al., 2019) |
| Algoritmo genético (DEAP, `genetica`) | Población 6, torneo de 3, cruce BLX-α con α = 0.5 y probabilidad 0.7 (Eshelman y Schaffer, 1993), mutación gaussiana σ = 0.15 con probabilidad 0.3 por individuo y 1/d por gen, elitismo de 1. Diversidad por generación: distancia media entre individuos y número de individuos distintos (Fortin et al., 2012) |
| Successive halving (extra) | Fidelidad: fracción de filas del entrenamiento interno. Factor η = 3 (Li et al., 2018). Rondas con 1/9, 1/3 y todos los datos (Jamieson y Talwalkar, 2016) |

Con un presupuesto de 30, el algoritmo genético alcanza unas 5 generaciones. Es poco para un algoritmo genético y es parte de lo que se compara:
con presupuestos chicos, un método que necesita muchas generaciones tiene desventaja.

In [4]:
plan = plan_rondas(config.PRESUPUESTO)
pd.DataFrame([{"ronda": i + 1, "candidatos": n, "fracción de datos": f, "costo en evaluaciones completas": n * f}
              for i, (n, f) in enumerate(plan)]).set_index("ronda")

,candidatos,fracción de datos,costo en evaluaciones completas
ronda,,,
1,90,0.111111,10.0
2,30,0.333333,10.0
3,10,1.000000,10.0


## 6. Presupuesto y piloto de tiempos

Cada modelo usa un solo hilo y el paralelismo va por combinación y pliegue externo (15 procesos). Así, además, el tiempo por evaluación
es comparable entre optimizadores.

Un primer piloto midió 4 evaluaciones de cada modelo en el primer pliegue externo. El costo total estimado fue de 224 horas-núcleo,
unas 15 horas. Un segundo piloto, en un pliegue interno de 31 mil filas, encontró dónde estaba el costo:

| Prueba | Tiempo | F1 macro |
|---|---|---|
| KNN k=15, KD-Tree, euclidiana | 4.72 s | 0.793 |
| KNN k=15, fuerza bruta, euclidiana | 0.28 s | 0.793 |
| KNN k=15, fuerza bruta, Manhattan | 4.52 s | 0.788 |
| Logística L1 con saga, C = 1, tolerancia 1e-4 | 5.35 s | 0.594 |
| Logística L1 con saga, C = 1, tolerancia 1e-3 | 0.61 s | 0.594 |
| Random Forest, 300 árboles, todas las filas | 14.65 s | 0.897 |
| Random Forest, 150 árboles, la mitad de las filas | 4.53 s | 0.897 |

Con 15 variables, el KD-Tree pierde contra la fuerza bruta, que usa álgebra lineal optimizada. Con esos cambios, el costo estimado baja a
unas 80 horas-núcleo. La comparación completa de algoritmos de KNN (fuerza bruta, KD-Tree, Ball-Tree y FAISS) va en el cuaderno computacional.

## 7. Estado de la tabla maestra

`runner.consolidar` junta lo que haya en `runs/partes/` y `runner.resumen` saca la media y la desviación entre pliegues externos de cada
combinación. Los resultados se analizan en los cuadernos siguientes; aquí solo se muestra el estado de la corrida. El conteo de 150
incluye las 14 corridas de successive halving, que no están entre las 140: son 136 combinaciones más 14 de successive halving.

In [6]:
aplicables, no_aplicables = runner.combinaciones()
print(f"Combinaciones aplicables: {len(aplicables)} | no aplicables: {len(no_aplicables)} | total: {len(aplicables) + len(no_aplicables)}")
maestra = runner.consolidar()
res = runner.resumen(maestra)
completas = res[res.pliegues == config.K_EXTERNO]
print(f"Combinaciones con los 5 pliegues: {len(completas)} de {len(aplicables)}")
completas.groupby(["tarea", "modelo"]).size().rename("combinaciones completas").to_frame()

Combinaciones aplicables: 136 | no aplicables: 4 | total: 140


Combinaciones con los 5 pliegues: 150 de 136


combinaciones completas
tarea modelo                                      
clf   KNN                                       13
      Naive Bayes                               17
      Random Forest                             17
      Regresión logística                       17
      SVM RBF                                   17
      XGBoost                                   17
      Árbol de decisión                         17
reg   KNN                                        5
      Lasso                                      5
      Random Forest                              5
      Ridge                                      5
      SVR RBF                                    5
      XGBoost                                    5
      Árbol de decisión                          5

## Referencias

- Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. En *Proceedings of the 25th ACM SIGKDD* (pp. 2623-2631). https://doi.org/10.1145/3292500.3330701
- Bergstra, J., Bardenet, R., Bengio, Y. y Kégl, B. (2011). Algorithms for hyper-parameter optimization. *Advances in Neural Information Processing Systems, 24*.
- Bergstra, J. y Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281-305.
- Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079-2107.
- Chawla, N. V., Bowyer, K. W., Hall, L. O. y Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research, 16*, 321-357. https://doi.org/10.1613/jair.953
- Eshelman, L. J. y Schaffer, J. D. (1993). Real-coded genetic algorithms and interval-schemata. En *Foundations of Genetic Algorithms, 2* (pp. 187-202). https://doi.org/10.1016/B978-0-08-094832-4.50018-0
- Fortin, F.-A., De Rainville, F.-M., Gardner, M.-A., Parizeau, M. y Gagné, C. (2012). DEAP: Evolutionary algorithms made easy. *Journal of Machine Learning Research, 13*, 2171-2175.
- He, H., Bai, Y., Garcia, E. A. y Li, S. (2008). ADASYN: Adaptive synthetic sampling approach for imbalanced learning. En *2008 IEEE International Joint Conference on Neural Networks* (pp. 1322-1328). https://doi.org/10.1109/IJCNN.2008.4633969
- Hsu, C.-W., Chang, C.-C. y Lin, C.-J. (2003). *A practical guide to support vector classification* (informe técnico). National Taiwan University.
- Jamieson, K. y Talwalkar, A. (2016). Non-stochastic best arm identification and hyperparameter optimization. En *Proceedings of AISTATS* (PMLR 51, pp. 240-248).
- Lemaître, G., Nogueira, F. y Aridas, C. K. (2017). Imbalanced-learn: A Python toolbox to tackle the curse of imbalanced datasets in machine learning. *Journal of Machine Learning Research, 18*(17), 1-5.
- Li, L., Jamieson, K., DeSalvo, G., Rostamizadeh, A. y Talwalkar, A. (2018). Hyperband: A novel bandit-based approach to hyperparameter optimization. *Journal of Machine Learning Research, 18*(185), 1-52.
- Probst, P., Boulesteix, A.-L. y Bischl, B. (2019). Tunability: Importance of hyperparameters of machine learning algorithms. *Journal of Machine Learning Research, 20*(53), 1-32.
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881
- Varma, S. y Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics, 7*, 91. https://doi.org/10.1186/1471-2105-7-91